In [1]:
import numpy as np
import nbimporter
from matplotlib import pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import gamma, norm
from sklearn.metrics import roc_auc_score
import os
import random
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from cdt.causality import pairwise
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import rankdata
from sklearn.linear_model import Lasso
import cepairsimplementation as ce
from scipy import stats
seedR = random.Random(42)
seedN = np.random.default_rng()

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


Detecting 1 CUDA device(s).


In [2]:
class MutualInfoNet(nn.Module):
    def __init__(self, input_dim_x, input_dim_y, hidden_dim):
        super(MutualInfoNet, self).__init__()
        self.fx = nn.Sequential(
            nn.Linear(input_dim_x, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        self.fy = nn.Sequential(
            nn.Linear(input_dim_y, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, y):
        z1 = self.fx(x)        # z1 = f(x)
        z2 = self.fy(y)        # z2 = g(y)
        s1 = z1
        s2 = z2 - z1           # s2 = g(y) - f(x)
        return s1, s2

In [3]:
def gaussian_kernel_matrix(x, y, sigma):
    """
    Computes the Gaussian kernel between all pairs (x_i, y_j)
    x: Tensor of shape (N, d)
    y: Tensor of shape (N, d)
    returns: Tensor of shape (N, N)
    """
    N, d = x.shape
    x = x.unsqueeze(1)  # (N, 1, d)
    y = y.unsqueeze(0)  # (1, N, d)
    diff = x - y
    dist_sq = (diff ** 2).sum(dim=2)  # (N, N)
    kernel = torch.exp(-dist_sq / sigma**2)
    return kernel

def mutual_information_loss(s1, s2, sigma=1.0, mu=1e-3, eps=1e-12):
    """
    Estimates mutual information between s1 and s2.
    s1, s2: Tensors of shape (N, d)
    returns: Scalar tensor representing -MI(s1, s2)
    """
    N = s1.shape[0]

    # Joint: k((s1_i, s2_i), (s1_j, s2_j))
    joint = torch.cat([s1, s2], dim=1)
    A = gaussian_kernel_matrix(joint, joint, sigma).mean(dim=1) + eps

    # Marginals
    B = gaussian_kernel_matrix(s1, s1, sigma).mean(dim=1) + eps
    C = gaussian_kernel_matrix(s2, s2, sigma).mean(dim=1) + eps

    mi_estimate = torch.mean(torch.log(A / (B * C)))
    s2_penalty = torch.mean(torch.sum(s2**2, dim=1))
    loss= mi_estimate + mu * s2_penalty
    return loss


In [4]:
def train_mi_network(model, x_data, y_data, epochs=1000, lr=1e-5, sigma=1.0,mu=1e-3):
    """
    Trains the model to maximize mutual information between s1 and s2.

    x_data, y_data: torch tensors of shape (N, input_dim)
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        s1, s2 = model(x_data, y_data)
        loss = mutual_information_loss(s1, s2, sigma=sigma, mu=mu)

        loss.backward()
        optimizer.step()
        
    return loss.item()

def pnl(d,hidden_dim = 16,mu=1e-3,epochs=100):
    x,y=d
    if x.shape[1]>1 or y.shape[1]>1:
        return np.nan
    x=ce.minmax_scale(x)
    y=ce.minmax_scale(y)
    x = torch.tensor(x, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.float32)

    #direction 1
    input_dim_x = x.shape[1]
    input_dim_y = y.shape[1]
    model = MutualInfoNet(input_dim_x=input_dim_x, input_dim_y=input_dim_y, hidden_dim=hidden_dim)
    lossx=train_mi_network(model, x, y, mu=mu,epochs=epochs) #if x the cause

    #direction 2
    input_dim_x = x.shape[1]
    input_dim_y = y.shape[1]
    model = MutualInfoNet(input_dim_x=input_dim_y, input_dim_y=input_dim_x, hidden_dim=hidden_dim)
    lossy=train_mi_network(model, y, x, mu=mu,epochs=epochs) #if y the cause
    #print("result",lossy-lossx)
    return lossy-lossx #if x the cause, then lossx<lossy, else lossy<lossx

In [ ]:
print(ce.test_tuebingen(pnl))